In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import sys
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

print("Python:", sys.version.split()[0])

for ten_thu_vien in ["pandas", "torch", "transformers", "tokenizers"]:
    try:
        print(ten_thu_vien, ":", version(ten_thu_vien))
    except PackageNotFoundError:
        print(ten_thu_vien, ": chưa cài")

thu_muc_du_lieu = Path("/content/drive/MyDrive/CMU-MOSI")

duong_dan_manifest = (
    thu_muc_du_lieu / "preprocessing" / "shared" / "manifest.csv"
)

duong_dan_pilot = (
    thu_muc_du_lieu / "preprocessing" / "shared" / "pilot_manifest.csv"
)

for duong_dan in [duong_dan_manifest, duong_dan_pilot]:
    print(duong_dan)
    print("Tồn tại:", duong_dan.is_file())

Python: 3.13.18
pandas : 2.2.3
torch : 2.11.0+cpu
transformers : 5.21.0
tokenizers : 0.23.3
/content/drive/MyDrive/CMU-MOSI/preprocessing/shared/manifest.csv
Tồn tại: True
/content/drive/MyDrive/CMU-MOSI/preprocessing/shared/pilot_manifest.csv
Tồn tại: True


In [3]:
import pandas as pd

kieu_cot = {
    "id": "string",
    "video_id": "string",
    "clip_id": "string",
}

manifest = pd.read_csv(
    duong_dan_manifest,
    dtype=kieu_cot,
    keep_default_na=False,
)

pilot = pd.read_csv(
    duong_dan_pilot,
    dtype=kieu_cot,
    keep_default_na=False,
)

assert len(manifest) == 2199, "Số mẫu manifest chưa đúng"
assert manifest["id"].is_unique, "Manifest có ID trùng"
assert pilot["id"].is_unique, "Pilot có ID trùng"

id_tao_lai = manifest["video_id"] + "$_$" + manifest["clip_id"]
assert manifest["id"].eq(id_tao_lai).all(), "Có ID sai quy tắc"

assert pilot["id"].isin(manifest["id"]).all(), "Pilot có ID ngoài manifest"

so_mau_theo_split = manifest["split"].value_counts().to_dict()
assert so_mau_theo_split == {
    "train": 1284,
    "valid": 229,
    "test": 686,
}, "Số mẫu từng split chưa đúng"

assert manifest["raw_text"].str.strip().ne("").all(), "Có transcript trống"

print("Số mẫu manifest:", len(manifest))
print("Số mẫu pilot:", len(pilot))
print("Phân bố split:", so_mau_theo_split)

mau_thu = pilot.iloc[0]

print("\nID thử:", mau_thu["id"])
print("Transcript gốc:", repr(mau_thu["raw_text"]))
print("Nhãn:", mau_thu["label"])

Số mẫu manifest: 2199
Số mẫu pilot: 10
Phân bố split: {'train': 1284, 'test': 686, 'valid': 229}

ID thử: 7JsX8y1ysxY$_$12
Transcript gốc: 'I DONT KNOW CLASSIC SUPERHERO KIND OF GETTING ON KIND OF THING'
Nhãn: 0.6


In [4]:
import importlib.util
import numpy as np

duong_dan_module = (
    thu_muc_du_lieu / "src" / "preprocessing" / "text_processor.py"
)

assert duong_dan_module.is_file(), "Không tìm thấy module Text"

dac_ta = importlib.util.spec_from_file_location(
    "mosi_text_processor",
    duong_dan_module,
)

text_module = importlib.util.module_from_spec(dac_ta)
dac_ta.loader.exec_module(text_module)

kiem_tra_ky_tu = text_module.kiem_tra_ky_tu
lam_sach_text = text_module.lam_sach_text
process_text = text_module.process_text

print("Đã nạp module:", duong_dan_module)

Đã nạp module: /content/drive/MyDrive/CMU-MOSI/src/preprocessing/text_processor.py


In [5]:
raw_text = mau_thu["raw_text"]

canh_bao = kiem_tra_ky_tu(raw_text)
clean_text = lam_sach_text(raw_text)

print("ID:", mau_thu["id"])
print("Raw text:  ", repr(raw_text))
print("Clean text:", repr(clean_text))
print("Có thay đổi:", raw_text != clean_text)
print("Số ký tự cần kiểm tra:", len(canh_bao))

if canh_bao:
    display(pd.DataFrame(canh_bao))

ID: 7JsX8y1ysxY$_$12
Raw text:   'I DONT KNOW CLASSIC SUPERHERO KIND OF GETTING ON KIND OF THING'
Clean text: 'I DONT KNOW CLASSIC SUPERHERO KIND OF GETTING ON KIND OF THING'
Có thay đổi: False
Số ký tự cần kiểm tra: 0


In [6]:
text_vi_du = "  I\tDONT   like\nthis this movie!!!  "

ket_qua = lam_sach_text(text_vi_du)

print("Trước:", repr(text_vi_du))
print("Sau:  ", repr(ket_qua))

assert ket_qua == "I DONT like this this movie!!!"

print("Đã giữ nguyên nội dung và chuẩn hóa khoảng trắng.")

Trước: '  I\tDONT   like\nthis this movie!!!  '
Sau:   'I DONT like this this movie!!!'
Đã giữ nguyên nội dung và chuẩn hóa khoảng trắng.


In [7]:
from transformers import AutoTokenizer

ten_checkpoint = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(
    ten_checkpoint,
    use_fast=True,
)

assert tokenizer.is_fast, "Cần fast tokenizer để lấy word_ids"

print("Checkpoint:", ten_checkpoint)
print("Loại tokenizer:", type(tokenizer).__name__)
print("Fast tokenizer:", tokenizer.is_fast)
print("Kích thước từ điển:", tokenizer.vocab_size)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Checkpoint: bert-base-uncased
Loại tokenizer: BertTokenizer
Fast tokenizer: True
Kích thước từ điển: 30522


In [8]:
words = clean_text.split()

ma_hoa = tokenizer(
    words,
    is_split_into_words=True,
    add_special_tokens=True,
    padding=False,
    truncation=False,
    return_attention_mask=True,
    return_token_type_ids=True,
)

input_ids = ma_hoa["input_ids"]
attention_mask = ma_hoa["attention_mask"]
token_type_ids = ma_hoa["token_type_ids"]

word_ids = [
    -1 if chi_so is None else chi_so
    for chi_so in ma_hoa.word_ids()
]

tokens = tokenizer.convert_ids_to_tokens(input_ids)

print("Clean text:", clean_text)
print("Words:", words)
print("Tokens:", tokens)
print("Input IDs:", input_ids)
print("Attention mask:", attention_mask)
print("Token type IDs:", token_type_ids)
print("Word IDs:", word_ids)

Clean text: I DONT KNOW CLASSIC SUPERHERO KIND OF GETTING ON KIND OF THING
Words: ['I', 'DONT', 'KNOW', 'CLASSIC', 'SUPERHERO', 'KIND', 'OF', 'GETTING', 'ON', 'KIND', 'OF', 'THING']
Tokens: ['[CLS]', 'i', 'don', '##t', 'know', 'classic', 'superhero', 'kind', 'of', 'getting', 'on', 'kind', 'of', 'thing', '[SEP]']
Input IDs: [101, 1045, 2123, 2102, 2113, 4438, 16251, 2785, 1997, 2893, 2006, 2785, 1997, 2518, 102]
Attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Token type IDs: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Word IDs: [-1, 0, 1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, -1]


In [9]:
bang_token = pd.DataFrame({
    "vi_tri_token": range(len(input_ids)),
    "token": tokens,
    "input_id": input_ids,
    "attention_mask": attention_mask,
    "token_type_id": token_type_ids,
    "word_id": word_ids,
    "tu_goc": [
        words[chi_so] if chi_so >= 0 else "(token đặc biệt)"
        for chi_so in word_ids
    ],
})

display(bang_token)

so_token = len(input_ids)

assert (
    so_token
    == len(attention_mask)
    == len(token_type_ids)
    == len(word_ids)
)

assert all(gia_tri == 1 for gia_tri in attention_mask)
assert all(gia_tri == 0 for gia_tri in token_type_ids)

assert all(
    chi_so == -1 or 0 <= chi_so < len(words)
    for chi_so in word_ids
)

tu_da_anh_xa = {
    chi_so for chi_so in word_ids
    if chi_so >= 0
}

assert tu_da_anh_xa == set(range(len(words))), (
    "Có từ chưa được ánh xạ sang token, cần kiểm tra"
)

print("Số từ:", len(words))
print("Số token, gồm token đặc biệt:", so_token)
print("Các kiểm tra đầu ra đã vượt qua.")

,vi_tri_token,token,input_id,attention_mask,token_type_id,word_id,tu_goc
0,0,[CLS],101,1,0,-1,(token đặc biệt)
1,1,i,1045,1,0,0,I
2,2,don,2123,1,0,1,DONT
3,3,##t,2102,1,0,1,DONT
4,4,know,2113,1,0,2,KNOW
5,5,classic,4438,1,0,3,CLASSIC
6,6,superhero,16251,1,0,4,SUPERHERO
7,7,kind,2785,1,0,5,KIND
8,8,of,1997,1,0,6,OF
9,9,getting,2893,1,0,7,GETTING


Số từ: 12
Số token, gồm token đặc biệt: 15
Các kiểm tra đầu ra đã vượt qua.


In [10]:
import numpy as np

config_text = {
    "checkpoint": ten_checkpoint,
    "tokenizer": tokenizer,
    "extract_embeddings": False,
}

In [11]:
ket_qua_text = process_text(
    mau_thu["raw_text"],
    config_text,
)

print("ID:", mau_thu["id"])
print("Clean text:", ket_qua_text["clean_text"])
print("Số từ:", len(ket_qua_text["words"]))

for ten_truong in [
    "input_ids",
    "attention_mask",
    "token_type_ids",
    "word_ids",
]:
    mang = ket_qua_text[ten_truong]

    print(
        ten_truong,
        "| shape:", mang.shape,
        "| dtype:", mang.dtype,
    )

print("Embeddings:", ket_qua_text["embeddings"])
print("Warnings:", ket_qua_text["warnings"])

assert ket_qua_text["raw_text"] == raw_text
assert ket_qua_text["clean_text"] == clean_text
assert ket_qua_text["words"] == words

np.testing.assert_array_equal(
    ket_qua_text["input_ids"],
    input_ids,
)

np.testing.assert_array_equal(
    ket_qua_text["attention_mask"],
    attention_mask,
)

np.testing.assert_array_equal(
    ket_qua_text["token_type_ids"],
    token_type_ids,
)

np.testing.assert_array_equal(
    ket_qua_text["word_ids"],
    word_ids,
)

print("Kết quả của hàm khớp với các ô đã chạy trước đó.")

ID: 7JsX8y1ysxY$_$12
Clean text: I DONT KNOW CLASSIC SUPERHERO KIND OF GETTING ON KIND OF THING
Số từ: 12
input_ids | shape: (15,) | dtype: int64
attention_mask | shape: (15,) | dtype: int64
token_type_ids | shape: (15,) | dtype: int64
word_ids | shape: (15,) | dtype: int64
Embeddings: None
Warnings: []
Kết quả của hàm khớp với các ô đã chạy trước đó.


In [12]:
from time import perf_counter

ket_qua_pilot = {}
nhat_ky_pilot = []

bat_dau_pilot = perf_counter()

for _, mau in pilot.iterrows():
    id_mau = mau["id"]
    bat_dau_mau = perf_counter()

    try:
        ket_qua = process_text(
            mau["raw_text"],
            config_text,
        )

        for ten_truong in [
            "input_ids",
            "attention_mask",
            "token_type_ids",
            "word_ids",
        ]:
            mang = ket_qua[ten_truong]

            if mang.ndim != 1 or mang.dtype != np.int64:
                raise ValueError(
                    f"{ten_truong} phải là mảng 1 chiều, kiểu int64"
                )

        if not np.all(ket_qua["attention_mask"] == 1):
            raise ValueError("Attention mask chưa đúng với chuỗi chưa padding")

        if not np.all(ket_qua["token_type_ids"] == 0):
            raise ValueError("Token type IDs chưa đúng với đầu vào một đoạn")

        ket_qua_pilot[id_mau] = ket_qua

        nhat_ky_pilot.append({
            "id": id_mau,
            "split": mau["split"],
            "status": "success",
            "so_tu": len(ket_qua["words"]),
            "so_token": len(ket_qua["input_ids"]),
            "text_thay_doi": (
                ket_qua["raw_text"] != ket_qua["clean_text"]
            ),
            "so_canh_bao": len(ket_qua["warnings"]),
            "thoi_gian_giay": perf_counter() - bat_dau_mau,
            "error": "",
        })

    except Exception as loi:
        nhat_ky_pilot.append({
            "id": id_mau,
            "split": mau["split"],
            "status": "failed",
            "so_tu": None,
            "so_token": None,
            "text_thay_doi": None,
            "so_canh_bao": None,
            "thoi_gian_giay": perf_counter() - bat_dau_mau,
            "error": f"{type(loi).__name__}: {loi}",
        })

tong_thoi_gian = perf_counter() - bat_dau_pilot

bang_nhat_ky = pd.DataFrame(nhat_ky_pilot)

display(bang_nhat_ky)

print("Tổng thời gian xử lý pilot:", round(tong_thoi_gian, 4), "giây")

,id,split,status,so_tu,so_token,text_thay_doi,so_canh_bao,thoi_gian_giay,error
0,7JsX8y1ysxY$_$12,train,success,12,15,False,0,0.000860,
1,8OtFthrtaJM$_$23,train,success,24,30,False,0,0.000792,
2,8d-gEyoeBzc$_$5,train,success,19,22,False,0,0.000658,
3,9T9Hf74oK10$_$9,train,success,20,22,False,0,0.000575,
4,Bfr499ggo-0$_$12,train,success,9,12,False,0,0.000436,
5,I5y0__X72p0$_$35,train,success,6,8,False,0,0.000380,
6,I5y0__X72p0$_$6,train,success,18,20,False,0,0.000508,
7,Nzq88NnDkEk$_$32,train,success,9,11,False,0,0.000535,
8,VbQk4H8hgr0$_$24,train,success,8,10,False,0,0.000363,
9,Vj1wYRQjB-o$_$2,train,success,21,23,False,0,0.000532,


Tổng thời gian xử lý pilot: 0.0095 giây


In [13]:
assert len(bang_nhat_ky) == len(pilot)
assert bang_nhat_ky["id"].is_unique
assert set(bang_nhat_ky["id"]) == set(pilot["id"])

so_thanh_cong = int(
    (bang_nhat_ky["status"] == "success").sum()
)

so_that_bai = int(
    (bang_nhat_ky["status"] == "failed").sum()
)

assert len(ket_qua_pilot) == so_thanh_cong

print("Số mẫu pilot:", len(pilot))
print("Thành công:", so_thanh_cong)
print("Thất bại:", so_that_bai)

mau_thanh_cong = bang_nhat_ky[
    bang_nhat_ky["status"] == "success"
]

if not mau_thanh_cong.empty:
    print(
        "Số mẫu thay đổi khoảng trắng:",
        int(mau_thanh_cong["text_thay_doi"].sum()),
    )

    print(
        "Số mẫu có cảnh báo ký tự:",
        int((mau_thanh_cong["so_canh_bao"] > 0).sum()),
    )

    print(
        "Số token ít nhất / nhiều nhất:",
        int(mau_thanh_cong["so_token"].min()),
        "/",
        int(mau_thanh_cong["so_token"].max()),
    )

mau_loi = bang_nhat_ky[
    bang_nhat_ky["status"] == "failed"
]

if not mau_loi.empty:
    print("\nCác mẫu thất bại:")
    display(mau_loi[["id", "error"]])

for id_mau, ket_qua in ket_qua_pilot.items():
    if ket_qua["warnings"]:
        print("\nCảnh báo của ID:", id_mau)
        display(pd.DataFrame(ket_qua["warnings"]))

Số mẫu pilot: 10
Thành công: 10
Thất bại: 0
Số mẫu thay đổi khoảng trắng: 0
Số mẫu có cảnh báo ký tự: 0
Số token ít nhất / nhiều nhất: 8 / 30


In [14]:
import json
from datetime import datetime, timezone
from importlib.metadata import version

ma_lan_chay = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S_%fZ"
)

thu_muc_luu = (
    thu_muc_du_lieu
    / "preprocessing"
    / "text"
    / "pilot"
    / ma_lan_chay
)

thu_muc_luu.mkdir(parents=True, exist_ok=False)

duong_dan_ket_qua = thu_muc_luu / "text_pilot_results.json"
duong_dan_nhat_ky = thu_muc_luu / "text_pilot_log.csv"
duong_dan_cau_hinh = thu_muc_luu / "text_pilot_config.json"

du_lieu_json = {}

for id_mau, ket_qua in ket_qua_pilot.items():
    du_lieu_json[id_mau] = {
        ten_truong: (
            gia_tri.tolist()
            if isinstance(gia_tri, np.ndarray)
            else gia_tri
        )
        for ten_truong, gia_tri in ket_qua.items()
    }

with duong_dan_ket_qua.open("w", encoding="utf-8") as tep:
    json.dump(
        du_lieu_json,
        tep,
        ensure_ascii=False,
        indent=2,
        allow_nan=False,
    )

bang_nhat_ky.to_csv(
    duong_dan_nhat_ky,
    index=False,
    encoding="utf-8-sig",
)

cau_hinh_luu = {
    "checkpoint": config_text["checkpoint"],
    "tokenizer_class": type(tokenizer).__name__,
    "word_split": "whitespace",
    "cleaning": "normalize_whitespace_only",
    "padding": False,
    "truncation": False,
    "extract_embeddings": False,
    "special_token_word_id": -1,
    "array_dtype": "int64",
    "pilot_ids": pilot["id"].tolist(),
    "success_count": so_thanh_cong,
    "failed_count": so_that_bai,
    "processing_seconds": tong_thoi_gian,
    "versions": {
        ten: version(ten)
        for ten in ["numpy", "pandas", "transformers", "tokenizers"]
    },
}

with duong_dan_cau_hinh.open("w", encoding="utf-8") as tep:
    json.dump(
        cau_hinh_luu,
        tep,
        ensure_ascii=False,
        indent=2,
        allow_nan=False,
    )

print("Đã lưu tại:", thu_muc_luu)

for duong_dan in [
    duong_dan_ket_qua,
    duong_dan_nhat_ky,
    duong_dan_cau_hinh,
]:
    print(duong_dan.name, "|", duong_dan.stat().st_size, "bytes")

Đã lưu tại: /content/drive/MyDrive/CMU-MOSI/preprocessing/text/pilot/20261102T092142_188135Z
text_pilot_results.json | 12677 bytes
text_pilot_log.csv | 748 bytes
text_pilot_config.json | 751 bytes


In [15]:
with duong_dan_ket_qua.open("r", encoding="utf-8") as tep:
    ket_qua_doc_lai = json.load(tep)

assert set(ket_qua_doc_lai) == set(ket_qua_pilot)

cac_truong_mang = [
    "input_ids",
    "attention_mask",
    "token_type_ids",
    "word_ids",
]

for id_mau, ban_goc in ket_qua_pilot.items():
    ban_doc_lai = ket_qua_doc_lai[id_mau]

    for ten_truong in cac_truong_mang:
        ban_doc_lai[ten_truong] = np.asarray(
            ban_doc_lai[ten_truong],
            dtype=np.int64,
        )

        np.testing.assert_array_equal(
            ban_doc_lai[ten_truong],
            ban_goc[ten_truong],
        )

    for ten_truong in [
        "raw_text",
        "clean_text",
        "words",
        "embeddings",
        "warnings",
    ]:
        assert ban_doc_lai[ten_truong] == ban_goc[ten_truong]

nhat_ky_doc_lai = pd.read_csv(
    duong_dan_nhat_ky,
    dtype={"id": "string"},
    keep_default_na=False,
)

assert nhat_ky_doc_lai["id"].tolist() == bang_nhat_ky["id"].tolist()
assert nhat_ky_doc_lai["status"].tolist() == bang_nhat_ky["status"].tolist()

print("Đọc lại thành công:", len(ket_qua_doc_lai), "kết quả Text")
print("Nội dung Text và các mảng khớp với trước khi lưu.")
print("ID và trạng thái trong nhật ký khớp.")

Đọc lại thành công: 10 kết quả Text
Nội dung Text và các mảng khớp với trước khi lưu.
ID và trạng thái trong nhật ký khớp.


In [16]:
import importlib.util

dac_ta = importlib.util.spec_from_file_location(
    "mosi_text_processor",
    duong_dan_module,
)

text_module = importlib.util.module_from_spec(dac_ta)
dac_ta.loader.exec_module(text_module)

so_mau_khop = 0

for _, mau in pilot.iterrows():
    id_mau = mau["id"]

    ket_qua_module = text_module.process_text(
        mau["raw_text"],
        config_text,
    )

    ket_qua_cu = ket_qua_pilot[id_mau]

    for ten_truong in [
        "input_ids",
        "attention_mask",
        "token_type_ids",
        "word_ids",
    ]:
        np.testing.assert_array_equal(
            ket_qua_module[ten_truong],
            ket_qua_cu[ten_truong],
        )

    for ten_truong in [
        "raw_text",
        "clean_text",
        "words",
        "embeddings",
        "warnings",
    ]:
        assert ket_qua_module[ten_truong] == ket_qua_cu[ten_truong], (
            f"Khác kết quả tại ID {id_mau}, trường {ten_truong}"
        )

    so_mau_khop += 1

print("Module:", text_module.__file__)
print("Số clip có kết quả khớp:", so_mau_khop, "/", len(pilot))

Module: /content/drive/MyDrive/CMU-MOSI/src/preprocessing/text_processor.py
Số clip có kết quả khớp: 10 / 10


In [17]:
import hashlib

noi_dung_py = duong_dan_module.read_bytes()

duong_dan_ban_sao = thu_muc_luu / "text_processor.py"

if duong_dan_ban_sao.exists():
    assert duong_dan_ban_sao.read_bytes() == noi_dung_py, (
        "Bản module trong thư mục pilot có nội dung khác"
    )
else:
    duong_dan_ban_sao.write_bytes(noi_dung_py)

ma_sha256 = hashlib.sha256(noi_dung_py).hexdigest()

print("Module gốc:", duong_dan_module)
print("Bản kèm pilot:", duong_dan_ban_sao)
print("SHA-256:", ma_sha256)
print("Đã lưu bản module khớp với module vừa kiểm chứng.")

Module gốc: /content/drive/MyDrive/CMU-MOSI/src/preprocessing/text_processor.py
Bản kèm pilot: /content/drive/MyDrive/CMU-MOSI/preprocessing/text/pilot/20261102T092142_188135Z/text_processor.py
SHA-256: 2f73f1dab3255d1bde99e05e3da8002e87f42ba8539738b88dbbe85c1b88a2ef
Đã lưu bản module khớp với module vừa kiểm chứng.


In [18]:
from pathlib import Path

duong_dan_module = Path(
    "/content/drive/MyDrive/CMU-MOSI/src/preprocessing/text_processor.py"
)

assert duong_dan_module.is_file(), "Không tìm thấy text_processor.py"

print(duong_dan_module.read_text(encoding="utf-8"))

import torch
"""Xử lý Text CMU-MOSI: làm sạch tối thiểu và tokenize."""

import unicodedata
import numpy as np


def kiem_tra_ky_tu(text):
    if not isinstance(text, str):
        raise TypeError("Transcript phải là chuỗi")

    canh_bao = []

    for vi_tri, ky_tu in enumerate(text):
        nhom = unicodedata.category(ky_tu)

        la_ky_tu_thay_the = ky_tu == "\ufffd"

        la_ky_tu_dieu_khien = (
            nhom in {"Cc", "Cf", "Cs"}
            and not ky_tu.isspace()
        )

        if la_ky_tu_thay_the or la_ky_tu_dieu_khien:
            canh_bao.append({
                "vi_tri": vi_tri,
                "ky_tu": repr(ky_tu),
                "ma_unicode": f"U+{ord(ky_tu):04X}",
                "nhom": nhom,
            })

    return canh_bao


def lam_sach_text(text):
    if not isinstance(text, str):
        raise TypeError("Transcript phải là chuỗi")

    clean_text = " ".join(text.split())

    if clean_text == "":
        raise ValueError("Transcript trống sau khi

In [19]:
import importlib.util
import torch
from transformers import AutoTokenizer, AutoModel

checkpoint = "bert-base-uncased"
thiet_bi = "cuda" if torch.cuda.is_available() else "cpu"

# Chỉ khởi tạo một lần, đặt ngoài vòng lặp xử lý clip
tokenizer_embedding = tokenizer

bert_co_dinh = AutoModel.from_pretrained(checkpoint)
bert_co_dinh.to(thiet_bi)
bert_co_dinh.eval()
bert_co_dinh.requires_grad_(False)

dac_ta = importlib.util.spec_from_file_location(
    "mosi_text_processor",
    duong_dan_module,
)

text_module = importlib.util.module_from_spec(dac_ta)
dac_ta.loader.exec_module(text_module)

config_embedding = {
    "tokenizer": tokenizer_embedding,
    "bert_model": bert_co_dinh,
    "extract_embeddings": True,
}

print("Thiết bị:", thiet_bi)
print("Đã khởi tạo BERT cố định.")

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Thiết bị: cpu
Đã khởi tạo BERT cố định.


In [20]:
import numpy as np

id_mau = str(pilot.iloc[0]["id"])
ban_cu = ket_qua_pilot[id_mau]

ket_qua_embedding = text_module.process_text(
    ban_cu["raw_text"],
    config_embedding,
)

# Thêm embedding không được làm đổi kết quả tokenize
for truong in [
    "input_ids",
    "attention_mask",
    "token_type_ids",
    "word_ids",
]:
    np.testing.assert_array_equal(
        ket_qua_embedding[truong],
        ban_cu[truong],
    )

# Kiểm tra khi tắt trích xuất
ket_qua_khong_embedding = text_module.process_text(
    ban_cu["raw_text"],
    {**config_embedding, "extract_embeddings": False},
)
assert ket_qua_khong_embedding["embeddings"] is None

print("ID:", id_mau)
print("Số token:", len(ket_qua_embedding["input_ids"]))
print("Embedding shape:", ket_qua_embedding["embeddings"].shape)
print("Embedding dtype:", ket_qua_embedding["embeddings"].dtype)
print("Tokenize khớp bản cũ; tắt embedding trả về None.")

ID: 7JsX8y1ysxY$_$12
Số token: 15
Embedding shape: (15, 768)
Embedding dtype: float32
Tokenize khớp bản cũ; tắt embedding trả về None.


In [21]:
import json
import hashlib
import numpy as np
import pandas as pd
from datetime import datetime, timezone
from importlib.metadata import version



thu_muc_text_pilot = thu_muc_du_lieu / "preprocessing" / "text" / "pilot"
tep_text = duong_dan_ket_qua
assert len(ket_qua_pilot) == len(pilot) == 10

ma_lan_chay = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S_%fZ"
)

thu_muc_embedding = (
    thu_muc_text_pilot / f"embeddings_{ma_lan_chay}"
)
thu_muc_embedding.mkdir(parents=True, exist_ok=False)

ket_qua_embedding_pilot = {}
nhat_ky_embedding = []

for id_mau in pilot["id"].astype(str):
    ban_cu = ket_qua_pilot[id_mau]

    ket_qua = text_module.process_text(
        ban_cu["raw_text"],
        config_embedding,
    )

    # Bổ sung embedding phải giữ nguyên kết quả Text
    for truong in ["raw_text", "clean_text", "words", "warnings"]:
        assert ket_qua[truong] == ban_cu[truong], (
            f"Khác trường {truong}: {id_mau}"
        )

    for truong in [
        "input_ids", "attention_mask",
        "token_type_ids", "word_ids",
    ]:
        np.testing.assert_array_equal(
            ket_qua[truong], ban_cu[truong]
        )

    embedding = ket_qua["embeddings"]
    assert embedding.shape == (
        len(ket_qua["input_ids"]),
        bert_co_dinh.config.hidden_size,
    )
    assert embedding.dtype == np.float32
    assert np.isfinite(embedding).all()

    tep_luu = thu_muc_embedding / f"{id_mau}.npz"

    np.savez_compressed(
        tep_luu,
        embeddings=embedding,
        input_ids=ket_qua["input_ids"],
        attention_mask=ket_qua["attention_mask"],
        token_type_ids=ket_qua["token_type_ids"],
        word_ids=ket_qua["word_ids"],
    )

    # Kiểm tra file đọc lại khớp dữ liệu trước khi lưu
    with np.load(tep_luu, allow_pickle=False) as du_lieu:
        for truong in du_lieu.files:
            np.testing.assert_array_equal(
                du_lieu[truong], ket_qua[truong]
            )

    ket_qua_embedding_pilot[id_mau] = ket_qua

    nhat_ky_embedding.append({
        "id": id_mau,
        "token_count": len(ket_qua["input_ids"]),
        "embedding_shape": str(embedding.shape),
        "dtype": str(embedding.dtype),
        "status": "success",
        "file": tep_luu.name,
    })

# Kiểm tra cùng đầu vào cho kết quả ổn định khi chạy lại
id_dau = str(pilot.iloc[0]["id"])
lap_lai = text_module.process_text(
    ket_qua_pilot[id_dau]["raw_text"],
    config_embedding,
)

np.testing.assert_allclose(
    lap_lai["embeddings"],
    ket_qua_embedding_pilot[id_dau]["embeddings"],
    rtol=1e-5,
    atol=1e-6,
)

# Lưu cấu hình để người khác biết cách tạo embedding
cau_hinh_luu = {
    "checkpoint": checkpoint,
    "model_revision": getattr(
        bert_co_dinh.config, "_commit_hash", None
    ),
    "layer": "last_hidden_state",
    "embedding_level": "token",
    "includes_special_tokens": True,
    "subword_pooling": None,
    "padding": False,
    "truncation": False,
    "frozen": True,
    "dtype": "float32",
    "device": str(next(bert_co_dinh.parameters()).device),
    "deepcu_input_config_confirmed": False,
    "text_results_source": str(tep_text),
    "module_sha256": hashlib.sha256(
        duong_dan_module.read_bytes()
    ).hexdigest(),
    "versions": {
        ten: version(ten)
        for ten in ["torch", "transformers", "numpy"]
    },
}

(thu_muc_embedding / "embedding_config.json").write_text(
    json.dumps(cau_hinh_luu, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

bang_embedding = pd.DataFrame(nhat_ky_embedding)
bang_embedding.to_csv(
    thu_muc_embedding / "embedding_log.csv",
    index=False,
    encoding="utf-8-sig",
)

display(bang_embedding)
print("Kết quả chạy lại ổn định.")
print("Đã lưu tại:", thu_muc_embedding)

,id,token_count,embedding_shape,dtype,status,file
0,7JsX8y1ysxY$_$12,15,"(15, 768)",float32,success,7JsX8y1ysxY$_$12.npz
1,8OtFthrtaJM$_$23,30,"(30, 768)",float32,success,8OtFthrtaJM$_$23.npz
2,8d-gEyoeBzc$_$5,22,"(22, 768)",float32,success,8d-gEyoeBzc$_$5.npz
3,9T9Hf74oK10$_$9,22,"(22, 768)",float32,success,9T9Hf74oK10$_$9.npz
4,Bfr499ggo-0$_$12,12,"(12, 768)",float32,success,Bfr499ggo-0$_$12.npz
5,I5y0__X72p0$_$35,8,"(8, 768)",float32,success,I5y0__X72p0$_$35.npz
6,I5y0__X72p0$_$6,20,"(20, 768)",float32,success,I5y0__X72p0$_$6.npz
7,Nzq88NnDkEk$_$32,11,"(11, 768)",float32,success,Nzq88NnDkEk$_$32.npz
8,VbQk4H8hgr0$_$24,10,"(10, 768)",float32,success,VbQk4H8hgr0$_$24.npz
9,Vj1wYRQjB-o$_$2,23,"(23, 768)",float32,success,Vj1wYRQjB-o$_$2.npz


Kết quả chạy lại ổn định.
Đã lưu tại: /content/drive/MyDrive/CMU-MOSI/preprocessing/text/pilot/embeddings_20261102T092204_055788Z


In [22]:
import shutil

tep_module_ban_giao = thu_muc_embedding / "text_processor.py"
shutil.copy2(duong_dan_module, tep_module_ban_giao)

noi_dung_huong_dan = f"""# Bàn giao Text pilot CMU-MOSI

## Giao diện
process_text(text, config) -> dict

## Xử lý Text
- Giữ raw_text; clean_text chỉ chuẩn hóa khoảng trắng, tab, xuống dòng.
- Giữ phủ định, từ đệm, từ lặp và dấu câu.
- Không stemming, xóa stopwords hoặc tự sửa transcript.
- Kiểm tra ký tự đáng ngờ và trả về warnings.

## Đầu ra
- raw_text, clean_text, words, warnings.
- input_ids, attention_mask, token_type_ids, word_ids:
  mảng int64 một chiều, cùng chiều dài.
- word_ids ánh xạ token về từ; token đặc biệt dùng -1.
- embeddings=None khi không yêu cầu trích xuất.

## Embedding pilot
- Checkpoint: bert-base-uncased.
- Lấy tầng cuối last_hidden_state theo token.
- Shape: (số token, 768), dtype float32.
- Gồm CLS và SEP; chưa gộp subword về từ.
- Khởi tạo BERT một lần ngoài vòng lặp.
- Dùng requires_grad_(False), eval() và inference_mode().
- Không padding hoặc truncation.
- Cấu hình đầu vào cuối cùng cho DeepCU chưa được nhóm xác nhận.

## File kết quả
- 10 file NPZ theo ID: embedding và các mảng token.
- embedding_config.json: cấu hình và phiên bản thư viện.
- embedding_log.csv: nhật ký xử lý.
- text_processor.py: bản module kèm kết quả.
- Text JSON gốc:
  {tep_text}
- Các đường dẫn trên tính từ thư mục CMU-MOSI.

## Kiểm chứng
- 10/10 mẫu pilot thành công.
- Kết quả tokenize khớp bản Text trước đó.
- File NPZ đọc lại khớp dữ liệu trước khi lưu.
- Một mẫu chạy lại cho kết quả ổn định.
- Chưa xử lý toàn bộ 2.199 mẫu.

## Alignment
Quy trình MFA nằm trong notebook MOSI_Alignment_Pilot.ipynb.
Clip 7JsX8y1ysxY$_$12 còn needs_review do chưa rõ cụm đầu I DONT KNOW.
"""

tep_huong_dan = thu_muc_embedding / "README_text_pilot.md"
tep_huong_dan.write_text(noi_dung_huong_dan, encoding="utf-8")

print("Đã lưu hướng dẫn:", tep_huong_dan)

Đã lưu hướng dẫn: /content/drive/MyDrive/CMU-MOSI/preprocessing/text/pilot/embeddings_20261102T092204_055788Z/README_text_pilot.md


In [23]:
%cd /content
!git clone https://github.com/danghau2512/DataMining-MSA.git

/content
Cloning into 'DataMining-MSA'...
remote: Enumerating objects: 57, done.
remote: Counting objects: 100% (57/57), done.
remote: Compressing objects: 100% (49/49), done.
remote: Total 57 (delta 19), reused 19 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (57/57), 8.25 MiB | 18.43 MiB/s, done.
Resolving deltas: 100% (19/19), done.


In [24]:
%cd /content/DataMining-MSA
!git status
!git branch -a

/content/DataMining-MSA
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
* main
  remotes/origin/HEAD -> origin/main
  remotes/origin/feat/preprocess-audio
  remotes/origin/feat/preprocess-visual
  remotes/origin/main


In [25]:
!git switch -c feat/preprocess-text

Switched to a new branch 'feat/preprocess-text'


In [26]:
from pathlib import Path

repo = Path("/content/DataMining-MSA")

print("CÁC MỤC TRONG REPOSITORY:")
for tep in sorted(repo.iterdir()):
    if not tep.name.startswith("."):
        print(tep.name)

print("\nCÁC NOTEBOOK TRÊN DRIVE:")
for tep in Path("/content/drive/MyDrive").rglob("MOSI*.ipynb"):
    print(tep)

print("\nMODULE TEXT:")
print(Path("/content/drive/MyDrive/CMU-MOSI/src/preprocessing/text_processor.py"))

print("\nCÁC FILE HƯỚNG DẪN:")
for tep in Path(
    "/content/drive/MyDrive/CMU-MOSI/preprocessing/text"
).rglob("README_text_pilot.md"):
    print(tep)

CÁC MỤC TRONG REPOSITORY:
README.md
src

CÁC NOTEBOOK TRÊN DRIVE:
/content/drive/MyDrive/Colab Notebooks/MOSI_Text_Preprocessing_BERT.ipynb
/content/drive/MyDrive/Colab Notebooks/MOSI_Alignment_Pilot.ipynb

MODULE TEXT:
/content/drive/MyDrive/CMU-MOSI/src/preprocessing/text_processor.py

CÁC FILE HƯỚNG DẪN:
/content/drive/MyDrive/CMU-MOSI/preprocessing/text/pilot/embeddings_20261101T185803_852622Z/README_text_pilot.md
/content/drive/MyDrive/CMU-MOSI/preprocessing/text/pilot/embeddings_20261102T092204_055788Z/README_text_pilot.md
